# 🐍 Day 32 · Python Basics — Enum, Settings, Logging & Tests (Production HITL Agent)

> **Goal of this notebook:** Production code needs clear statuses, settings from the environment, logs and tests. Learn the Python used in `src/` and `tests/` of today's project.

**How to use it**
1. Click on a grey code cell and press **Shift + Enter** to run it.
2. Read the comments (lines that start with `#`) — they explain every step.
3. Change the values and run the cell again. Breaking things is how you learn! 🙂

📦 **First** do *Step 1 - Install today's packages* in [`00_START_HERE.md`](00_START_HERE.md).

✅ No API key needed for this notebook.

### 📋 What you will learn
1. `Enum` for workflow statuses
2. A settings class that reads environment variables
3. Turning text into `True`/`False`
4. Logging instead of `print`
5. Custom exceptions and `finally`
6. Writing tests with `assert` (the pytest way)

## 1. `Enum` for statuses
`src/state.py` uses an Enum like this, so a typo such as `"aproved"` is impossible.

In [ ]:
from enum import Enum

class WorkflowStatus(str, Enum):        # str + Enum -> also behaves like a string
    DRAFTING = "drafting"
    AWAITING_REVIEW = "awaiting_review"
    APPROVED = "approved"
    PUBLISHED = "published"

status = WorkflowStatus.AWAITING_REVIEW
print(status, "|", status.value, "|", status == "awaiting_review")

## 2. Settings from the environment
`src/config.py` reads every setting from `.env` with a sensible default.

In [ ]:
import os

def to_bool(text):
    return str(text).strip().lower() in {"1", "true", "yes", "on"}

class Settings:
    def __init__(self):
        self.app_name = os.getenv("APP_NAME", "HITL LinkedIn Agent")
        self.port = int(os.getenv("PORT", "8000"))                     # text -> number
        self.simulation = to_bool(os.getenv("LINKEDIN_SIMULATION_MODE", "true"))
        self.temperature = float(os.getenv("LLM_TEMPERATURE", "0.7"))

settings = Settings()
print(vars(settings))      # vars() shows all attributes as a dict

## 3. Logging instead of `print`
Logs have a **level** (DEBUG < INFO < WARNING < ERROR) and a time stamp, and can be switched on or off without changing code.

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s", force=True)
log = logging.getLogger("hitl")

log.debug("you will NOT see this (level is INFO)")
log.info("Draft created for topic 'AI agents'")
log.warning("Reviewer has not answered for 24 hours")
log.error("LinkedIn API returned 401")

## 4. Custom exceptions and `finally`

In [ ]:
class PublishError(Exception):
    "Raised when a post cannot be published."

def publish(post, status):
    if status != WorkflowStatus.APPROVED:
        raise PublishError(f"Post must be approved first (status is {status.value})")
    return f"Published: {post[:30]}..."

for s in [WorkflowStatus.AWAITING_REVIEW, WorkflowStatus.APPROVED]:
    try:
        print(publish("Agents are changing software development", s))
    except PublishError as e:
        log.error(e)
    finally:
        print("   (finally always runs — good place to close connections)")

## 5. 🔗 Tests with `assert`
`assert condition` does nothing when the condition is `True` and raises an error when it is `False`.
`pytest` finds functions starting with `test_` and runs them. Here we run them ourselves.

In [ ]:
def word_limit_ok(post, limit=50):
    return len(post.split()) <= limit

def test_short_post_is_ok():
    assert word_limit_ok("Short post about AI") is True

def test_long_post_is_rejected():
    assert word_limit_ok("word " * 60) is False

def test_cannot_publish_unapproved():
    try:
        publish("x", WorkflowStatus.DRAFTING)
        assert False, "should have raised"
    except PublishError:
        pass

tests = [test_short_post_is_ok, test_long_post_is_rejected, test_cannot_publish_unapproved]
for t in tests:
    t()
    print("✅ passed:", t.__name__)
# In the project folder you would simply run:  uv run pytest

---
## 🏋️ Practice time
Try these on your own in the empty cell below:

1. Add a status `REJECTED = "rejected"` to the Enum.
2. Set `os.environ["PORT"] = "9000"`, create `Settings()` again and print the port.
3. Write a test `test_to_bool()` that checks `to_bool("Yes")` is `True` and `to_bool("0")` is `False`.

In [ ]:
# ✍️ Write your answers here

<details>
<summary>👀 Click here to see the solutions (try first!)</summary>

```python
class WorkflowStatus2(str, Enum):
    APPROVED = "approved"
    REJECTED = "rejected"

os.environ["PORT"] = "9000"
print(Settings().port)

def test_to_bool():
    assert to_bool("Yes") is True
    assert to_bool("0") is False
test_to_bool(); print("passed")
```
</details>

## ✅ Summary

| Concept | Example |
|---|---|
| Enum | `class Status(str, Enum): APPROVED = 'approved'` |
| Env with default | `os.getenv('PORT', '8000')` |
| Text -> bool | `text.lower() in {'1', 'true', 'yes'}` |
| Logger | `log = logging.getLogger('app'); log.info('...')` |
| Custom error | `class PublishError(Exception): ...` |
| Test | `def test_x(): assert f() == 1` |

## ➡️ What's next?
You now know the Python used in **Day_32_Production_HITL_LinkedIn_Agent**.

Open [`00_START_HERE.md`](00_START_HERE.md) and continue with **lesson 1**. 🚀